In [9]:
import pickle
import os
import pandas as pd
import numpy as np

def extract_all_network_metrics():
    # 1. Find all folders starting with 'data_'
    data_folders = [f for f in os.listdir('.') if os.path.isdir(f) and f.startswith('data_')]
    
    if not data_folders:
        print("ERROR: No folders starting with 'data_' found in the current directory.")
        return

    print(f"Found {len(data_folders)} data folders: {data_folders}\n")

    # 2. Loop through each folder and extract metrics
    for folder in data_folders:
        file_path = os.path.join(folder, "intervention_data.pkl")
        
        if not os.path.exists(file_path):
            print(f"Skipping {folder}: 'intervention_data.pkl' not found.")
            continue

        try:
            with open(file_path, "rb") as f:
                data = pickle.load(f)
            
            # The network stats are stored under the key 'network_df'
            # It might be a DataFrame or a list of dicts depending on how it was saved
            raw_stats = data.get("network_df")
            
            if raw_stats is None:
                print(f"[{folder}] No 'network_df' key found in pickle file.")
                continue

            # Ensure it is a DataFrame
            if isinstance(raw_stats, list):
                df = pd.DataFrame(raw_stats)
            else:
                df = pd.DataFrame(raw_stats) # Handles case where it's already a DF

            print(f"========================================")
            print(f" NETWORK METRICS FOR: {folder}")
            print(f"========================================")
            
            # 3. Calculate and print statistics for every numeric column
            # We exclude 'seed' or other non-metric columns if they exist
            metrics = [c for c in df.columns if c != "seed"]
            
            if not metrics:
                print("No metrics found in dataframe.")
            else:
                # Create a clean summary table
                summary = df[metrics].agg(['mean', 'std', 'min', 'max']).transpose()
                print(summary.round(4))
                
            print("\n")

        except Exception as e:
            print(f"Error processing {folder}: {e}")

if __name__ == "__main__":
    extract_all_network_metrics()

Found 12 data folders: ['data_BA', 'data_BA_late', 'data_BA_strong_b', 'data_BA_weak_b', 'data_ER', 'data_Grids', 'data_HighBetaI3.0', 'data_InitialAdoption0.3', 'data_InitialAdoption0.5', 'data_InitialInfrastructure0.15', 'data_InitialInfrastructure0.25', 'data_LowBetaI1.0']

 NETWORK METRICS FOR: data_BA
                            mean    std       min       max
avg_clustering            0.0957  0.029    0.0342    0.1902
avg_degree                3.9467  0.000    3.9467    3.9467
density                   0.0265  0.000    0.0265    0.0265
num_components            1.0000  0.000    1.0000    1.0000
largest_component_size  150.0000  0.000  150.0000  150.0000


 NETWORK METRICS FOR: data_BA_late
                            mean    std       min       max
avg_clustering            0.0957  0.029    0.0342    0.1902
avg_degree                3.9467  0.000    3.9467    3.9467
density                   0.0265  0.000    0.0265    0.0265
num_components            1.0000  0.000    1.0000    1.

In [ ]:
import pickle
import os
import pandas as pd
import numpy as np

def extract_combined_metrics():
    print("=== COMBINED METRICS: Network Structure + Adoption Speed & Probability ===\n")

    # 1. Find all folders starting with 'data_'
    data_folders = [f for f in os.listdir('.') if os.path.isdir(f) and f.startswith('data_')]
    
    if not data_folders:
        print("ERROR: No folders starting with 'data_' found.")
        return

    summary_rows = []

    # 2. Loop through each folder
    for folder in data_folders:
        file_path = os.path.join(folder, "intervention_data.pkl")
        
        if not os.path.exists(file_path):
            continue

        try:
            with open(file_path, "rb") as f:
                data = pickle.load(f)

            # =========================
            # PART A: NETWORK METRICS
            # =========================
            print(f"--- SCENARIO: {folder} ---")
            
            raw_stats = data.get("network_df")
            network_summary = {}

            if raw_stats is not None:
                # Ensure it's a DataFrame
                if isinstance(raw_stats, list):
                    df_net = pd.DataFrame(raw_stats)
                else:
                    df_net = pd.DataFrame(raw_stats)

                # Calculate means for key structural metrics
                # We save these to print and to export later
                if 'avg_clustering' in df_net.columns:
                    network_summary['Avg_Clustering'] = df_net['avg_clustering'].mean()
                if 'avg_degree' in df_net.columns:
                    network_summary['Avg_Degree'] = df_net['avg_degree'].mean()
                if 'largest_component_size' in df_net.columns:
                    network_summary['Max_Component'] = df_net['largest_component_size'].mean()

                print(" [Network Structure]")
                print(f"   Avg Clustering:    {network_summary.get('Avg_Clustering', 0):.4f}")
                print(f"   Avg Degree:        {network_summary.get('Avg_Degree', 0):.2f}")
                print(f"   Largest Component: {network_summary.get('Max_Component', 0):.1f}")
            else:
                print(" [Network Structure] No network data found.")

            # ======================================
            # PART B: ADOPTION SPEED & PROBABILITY 
            # =====================================
            print(" [Adoption Dynamics]")
            
            scenarios = {
                "Baseline": data.get("baseline_X"),
                "Subsidy":  data.get("subsidy_X")
            }

            for label, trajectories in scenarios.items():
                if trajectories is None:
                    continue

                # 1. Probability of High Adoption (>= 80%)
                final_values = [traj[-1] for traj in trajectories]
                success_count = sum(x >= 0.80 for x in final_values)
                total_trials = len(trajectories)
                prob_high = success_count / total_trials

                # 2. Speed to Stability (Time step crossing 80%)
                speeds = []
                for traj in trajectories:
                    # Get indices where adoption >= 0.80
                    high_share_indices = np.where(traj >= 0.80)[0]
                    if len(high_share_indices) > 0:
                        speeds.append(high_share_indices[0]) # First crossing
                
                # Calculate Avg Speed (only for successful trials)
                if speeds:
                    avg_speed = np.mean(speeds)
                    std_speed = np.std(speeds)
                else:
                    avg_speed = np.nan
                    std_speed = 0.0

                print(f"   {label:10} -> Prob(>=80%): {prob_high*100:5.1f}% | Speed to 80%: {avg_speed:5.1f} steps")
    

            print("-" * 40)

        except Exception as e:
            print(f"Error processing {folder}: {e}")


if __name__ == "__main__":
    extract_combined_metrics()

=== COMBINED METRICS: Network Structure + Adoption Speed & Probability ===

--- SCENARIO: data_BA ---
 [Network Structure]
   Avg Clustering:    0.0957
   Avg Degree:        3.95
   Largest Component: 150.0
 [Adoption Dynamics]
   Success Rate (>=80%): 49.5%
   Baseline   -> Prob(>=80%):  49.5% | Speed to 80%:   1.5 steps
   Subsidy    -> Prob(>=80%):  76.0% | Speed to 80%:   0.8 steps
----------------------------------------
--- SCENARIO: data_BA_late ---
 [Network Structure]
   Avg Clustering:    0.0957
   Avg Degree:        3.95
   Largest Component: 150.0
 [Adoption Dynamics]
   Success Rate (>=80%): 49.5%
   Baseline   -> Prob(>=80%):  49.5% | Speed to 80%:   1.5 steps
   Subsidy    -> Prob(>=80%):  54.5% | Speed to 80%:   4.5 steps
----------------------------------------
--- SCENARIO: data_BA_strong_b ---
 [Network Structure]
   Avg Clustering:    0.0957
   Avg Degree:        3.95
   Largest Component: 150.0
 [Adoption Dynamics]
   Success Rate (>=80%): 49.5%
   Baseline   -> Pr